In [3]:
# @title 🧪 Grid Search Optimizer (LaBSE + MuRIL)
# !pip install transformers sentence-transformers scipy scikit-learn pandas

import torch
import numpy as np
import pandas as pd
import ast
from scipy.optimize import linear_sum_assignment
from transformers import AutoTokenizer, AutoModel
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

# ==========================================================
# 1. LOAD DATA (Your 43 Corrected Sentences)
# ==========================================================
FINAL_DATASET_REFINED = [
    {"eng": "Now, we will look at the overview of Online Social Media.", "tam": "இப்போது நாம் Online Social Media பற்றிய ஒரு மேலோட்டத்தைப் பார்ப்போம்", "indices": "[(0,0), (1,1), (8,2), (9,3), (10,4), (7,5), (5,6), (6,7), (2,8), (3,8), (4,8)]"},
    {"eng": "Social media is of different types, and different types of content are getting generated in our social media.", "tam": "Social media பலவிதமானது, அதில் பலவிதமான உள்ளடக்கங்கள் உருவாகிக் கொண்டிருக்கின்றன", "indices": "[(0,0), (1,1), (4,2), (5,2), (7,4), (8,4), (10,5), (13,6), (11,7), (12,7)]"},
    {"eng": "There are also many different ways in which social media content is generated.", "tam": "மேலும், social media உள்ளடக்கங்கள் உருவாக பல்வேறு வழிகளும் உள்ளன", "indices": "[(2,0), (8,1), (9,2), (10,3), (12,4), (4,5), (5,6), (1,7), (0,7)]"},
    {"eng": "In this course, we will look at the different types of social media services that exist.", "tam": "இந்த course-இல், தற்போது உள்ள பல்வேறு வகையான social media சேவைகளைப் பற்றிப் பார்ப்போம்", "indices": "[(0,1), (1,0), (2,1), (3,0), (8,4), (9,5), (11,6), (12,7), (13,8), (15,3), (4,10), (5,10), (6,10)]"},
    {"eng": "We will also look at how to collect data from social media.", "tam": "மேலும், social media-விலிருந்து தரவுகளை எவ்வாறு சேகரிப்பது என்பது பற்றியும் பார்ப்போம்", "indices": "[(2,0), (10,1), (11,1), (9,1), (8,3), (5,4), (6,5), (7,5), (0,7), (1,7), (3,8), (4,8)]"},
    {"eng": "Specifically, we will look at APIs that we can use to collect data.", "tam": "குறிப்பாக, தரவுகளைச் சேகரிக்க நாம் பயன்படுத்தக்கூடிய API-களைப் பார்ப்போம்", "indices": "[(0,0), (12,1), (10,2), (11,2), (1,3), (7,3), (9,4), (5,5), (2,6), (3,6), (4,6)]"},
    {"eng": "We will specifically look at the Twitter API and the Facebook API.", "tam": "நாம் குறிப்பாக Twitter API மற்றும் Facebook API ஆகியவற்றைப் பார்ப்போம்", "indices": "[(0,0), (2,1), (6,2), (7,3), (8,4), (10,5), (11,6), (1,8), (3,8), (4,8)]"},
    {"eng": "We will see how to collect data from Twitter and Facebook.", "tam": "Twitter மற்றும் Facebook-லிருந்து தரவுகளை எவ்வாறு சேகரிப்பது என்பதை நாம் பார்ப்போம்", "indices": "[(8,0), (9,1), (10,2), (7,2), (6,3), (3,4), (4,5), (5,5), (0,7), (1,8), (2,8)]"},
    {"eng": "We will also look at how to process the collected data.", "tam": "சேகரிக்கப்பட்ட தரவுகளை எவ்வாறு செயல்முறைப்படுத்துவது என்பது பற்றியும் பார்ப்போம்", "indices": "[(9,0), (10,1), (5,2), (7,3), (2,5), (0,6), (1,6), (3,6), (4,6)]"},
    {"eng": "Since the data is text, we will have to look at text analysis techniques.", "tam": "தரவு text வடிவில் இருப்பதால், நாம் text analysis techniques-ஐப் பார்க்க வேண்டும்", "indices": "[(2,0), (4,1), (0,3), (3,3), (5,4), (11,5), (12,6), (13,7), (9,8), (10,8), (6,9), (7,9), (8,9)]"},
    {"eng": "Since the data is in the form of a graph or a network, we will have to use graph analysis techniques.", "tam": "தரவு ஒரு graph அல்லது network வடிவில் இருப்பதால், நாம் graph analysis techniques-ஐப் பயன்படுத்த வேண்டும்", "indices": "[(2,0), (9,2), (10,3), (12,4), (0,6), (13,7), (18,8), (19,9), (20,10), (17,11), (14,12), (15,12), (16,12)]"},
    {"eng": "So we will look at graph analysis metrics like degree, closeness, and betweenness centrality.", "tam": "எனவே, degree, closeness மற்றும் betweenness centrality போன்ற graph analysis metrics-ஐ நாம் பார்ப்போம்", "indices": "[(0,0), (9,1), (10,2), (11,3), (12,4), (13,5), (8,6), (5,7), (6,8), (7,9), (1,10), (2,11), (3,11), (4,11)]"},
    {"eng": "We will also look at how to analyze and visualize networks using a tool called Gephi.", "tam": "Gephi என்ற கருவியைப் பயன்படுத்தி networks-ஐ எவ்வாறு பகுப்பாய்வு செய்வது மற்றும் காட்சிப்படுத்துவது என்பது பற்றியும் பார்ப்போம்", "indices": "[(15,0), (13,2), (11,3), (10,4), (5,5), (7,6), (6,6), (8,8), (9,9), (2,11), (3,11), (1,12), (3,12)]"},
    {"eng": "In the end, we will look at two specific topics.", "tam": "இறுதியாக, நாம் இரண்டு குறிப்பிட்ட தலைப்புகளைப் பார்ப்போம்", "indices": "[(0,0), (1,0), (2,0), (3,1), (7,2), (8,3), (9,4), (4,5), (5,5), (6,5)]"},
    {"eng": "One is privacy, and the other is fake news.", "tam": "ஒன்று தனியுரிமை (privacy), மற்றொன்று போலிச் செய்திகள் (fake news)", "indices": "[(0,0), (2,1), (2,2), (5,3), (7,4), (8,5), (7,6), (8,7)]"},
    {"eng": "The course has tutorials on Python.", "tam": "இந்த course-இல் Python குறித்த tutorials உள்ளன", "indices": "[(0,1), (1,1), (5,2), (4,3), (3,4), (2,5)]"},
    {"eng": "It is not a prerequisite that you should know Python.", "tam": "உங்களுக்கு Python தெரிந்திருக்க வேண்டும் என்பது கட்டாயமில்லை", "indices": "[(6,0), (9,1), (8,2), (7,3), (2,5), (4,5)]"},
    {"eng": "But if you know Python, it will be really useful.", "tam": "ஆனால் உங்களுக்கு Python தெரிந்தால், அது மிகவும் பயனுள்ளதாக இருக்கும்", "indices": "[(0,0), (2,1), (4,2), (3,3), (1,3), (5,4), (9,6), (6,7), (7,7)]"},
    {"eng": "Tutorials on setting up Linux and Python are also uploaded.", "tam": "Linux மற்றும் Python-ஐ நிறுவுவதற்கான tutorials-உம் பதிவேற்றப்பட்டுள்ளன", "indices": "[(4,0), (5,1), (6,2), (2,3), (3,3), (1,3), (0,4), (8,4), (9,5)]"},
    {"eng": "These hands-on tutorials will help students write Python code to collect data from social media platforms", "tam": "இந்த hands-on tutorials மாணவர்களுக்கு Python code எழுதவும், social media platforms-இலிருந்து தரவு சேகரிக்கவும் உதவும்", "indices": "[(0,0), (1,1), (2,2), (5,3), (7,4), (8,5), (6,6), (13,7), (14,8), (15,9), (12,9), (11,10), (10,11), (9,11), (4,12)]"},
    {"eng": "Next week, we will briefly discuss one topic introduced today and provide more hands-on tutorials.", "tam": "அடுத்த வாரம், இன்று அறிமுகப்படுத்தப்பட்ட ஒரு topic-ஐ சிறிது விரிவாகப் பேசுவோம் மற்றும் கூடுதல் hands-on tutorials வழங்கப்படும்", "indices": "[(0,0), (1,1), (8,2), (7,3), (6,4), (5,5), (4,6), (2,7), (3,7), (9,8), (12,9), (13,10), (14,11), (10,12)]"},
    {"eng": "This course is on Online Social Media.", "tam": "இந்த course Online Social Media பற்றியது", "indices": "[(0,0), (1,1), (4,2), (5,3), (6,4), (2,5), (3,5)]"},
    {"eng": "I am Ponnurangam Kumaraguru.", "tam": "நான் பொன்னுரங்கம் குமரகுரு", "indices": "[(0,0), (2,1), (3,2)]"},
    {"eng": "I am a professor at IIIT Hyderabad.", "tam": "நான் IIIT Hyderabad-இல் பேராசிரியராக இருக்கிறேன்", "indices": "[(0,0), (5,1), (6,2), (4,2), (3,3), (1,4)]"},
    {"eng": "We will have weekly sessions where we will look at different topics.", "tam": "நாம் வாராந்திர அமர்வுகளைக் கொண்டிருப்போம், அதில் பல்வேறு தலைப்புகளைப் பார்ப்போம்", "indices": "[(0,0), (3,1), (4,2), (1,3), (2,3), (6,4), (10,5), (11,6), (8,7), (9,7)]"},
    {"eng": "I prepared these slides for a specific purpose.", "tam": "நான் ஒரு குறிப்பிட்ட நோக்கத்திற்காக இந்த slides-ஐ தயார் செய்தேன்", "indices": "[(0,0), (5,1), (6,2), (7,3), (4,3), (2,4), (3,5), (1,6), (1,7)]"},
    {"eng": "This book is very interesting.", "tam": "இந்த புத்தகம் மிகவும் சுவாரஸ்யமாக இருக்கிறது", "indices": "[(0,0), (1,1), (3,2), (4,3), (2,4)]"},
    {"eng": "We went to the market yesterday.", "tam": "நாங்கள் நேற்று சந்தைக்குச் சென்றோம்", "indices": "[(0,0), (5,1), (2,2), (3,2), (4,2), (1,3)]"},
    {"eng": "The cat is sleeping on the mat.", "tam": "பூனை பாயில் தூங்கிக்கொண்டிருக்கிறது", "indices": "[(1,0), (4,1), (5,1), (6,1), (2,2), (3,2)]"},
    {"eng": "He is playing cricket.", "tam": "அவன் கிரிக்கெட் விளையாடிக்கொண்டிருக்கிறான்", "indices": "[(0,0), (3,1), (1,2), (2,2)]"},
    {"eng": "She sings beautifully.", "tam": "அவள் அழகாகப் பாடுகிறாள்", "indices": "[(0,0), (2,1), (1,2)]"},
    {"eng": "They are learning Tamil.", "tam": "அவர்கள் தமிழ் கற்றுக்கொண்டிருக்கிறார்கள்", "indices": "[(0,0), (3,1), (1,2), (2,2)]"},
    {"eng": "The sun rises in the east.", "tam": "சூரியன் கிழக்கில் உதிக்கிறது", "indices": "[(1,0), (3,1), (4,1), (5,1), (2,2)]"},
    {"eng": "I like to drink coffee.", "tam": "எனக்கு காபி குடிக்க பிடிக்கும்", "indices": "[(0,0), (4,1), (2,2), (3,2), (1,3)]"},
    {"eng": "My friend is coming today.", "tam": "என் நண்பன் இன்று வருகிறான்", "indices": "[(0,0), (1,1), (4,2), (2,3), (3,3)]"},
    {"eng": "Please give me some water.", "tam": "தயவுசெய்து எனக்கு கொஞ்சம் தண்ணீர் கொடுங்கள்", "indices": "[(0,0), (2,1), (3,2), (4,3), (1,4)]"},
    {"eng": "We should respect our elders.", "tam": "நாம் பெரியவர்களை மதிக்க வேண்டும்", "indices": "[(0,0), (4,1), (3,1), (2,2), (1,3)]"},
    {"eng": "Education is important for everyone.", "tam": "கல்வி அனைவருக்கும் முக்கியமானது", "indices": "[(0,0), (3,1), (4,1), (2,2), (1,2)]"},
    {"eng": "He runs very fast.", "tam": "அவன் மிகவும் வேகமாக ஓடுகிறான்", "indices": "[(0,0), (2,1), (3,2), (1,3)]"},
    {"eng": "The dog is barking.", "tam": "நாய் குரைத்துக்கொண்டிருக்கிறது", "indices": "[(1,0), (2,1), (3,1)]"},
    {"eng": "I saw a movie yesterday.", "tam": "நான் நேற்று ஒரு படம் பார்த்தேன்", "indices": "[(0,0), (4,1), (2,2), (3,3), (1,4)]"},
    {"eng": "She is reading a book.", "tam": "அவள் ஒரு புத்தகம் படித்துக்கொண்டிருக்கிறாள்", "indices": "[(0,0), (3,1), (4,2), (1,3), (2,3)]"},
    {"eng": "They went to the beach.", "tam": "அவர்கள் கடற்கரைக்குச் சென்றார்கள்", "indices": "[(0,0), (2,1), (3,1), (4,1), (1,2)]"}
]

# ==========================================================
# 2. LOAD MODELS
# ==========================================================
print("⏳ Loading Models...")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# LaBSE
labse = SentenceTransformer('sentence-transformers/LaBSE').to(device)

# MuRIL
muril_id = "google/muril-base-cased"
tokenizer = AutoTokenizer.from_pretrained(muril_id)
muril = AutoModel.from_pretrained(muril_id, output_hidden_states=True).to(device)
muril.eval()
print("✅ Models Ready.")

# ==========================================================
# 3. PRE-COMPUTATION (Speeds up grid search 100x)
# ==========================================================
def get_muril_embeddings(text):
    words = text.split()
    inputs = tokenizer(words, is_split_into_words=True, return_tensors="pt", padding=True).to(device)
    with torch.no_grad(): out = muril(**inputs)
    # Sum last 4 layers
    hidden_states = torch.stack(out.hidden_states)
    word_embeddings = torch.sum(hidden_states[-4:], dim=0)
    # Subword pooling
    word_ids = inputs.word_ids()
    final_vectors = []
    current_vectors = []
    current_word_idx = None
    for i, word_idx in enumerate(word_ids):
        if word_idx is None: continue
        if current_word_idx is None:
            current_word_idx = word_idx
            current_vectors.append(word_embeddings[0, i])
        elif word_idx == current_word_idx:
            current_vectors.append(word_embeddings[0, i])
        else:
            final_vectors.append(torch.stack(current_vectors).mean(dim=0))
            current_word_idx = word_idx
            current_vectors = [word_embeddings[0, i]]
    if current_vectors: final_vectors.append(torch.stack(current_vectors).mean(dim=0))
    return torch.stack(final_vectors).cpu().numpy()

# Cache class to store matrices
class CachedPair:
    def __init__(self, src, tgt, gold_indices_str):
        self.gold_set = set(ast.literal_eval(gold_indices_str))

        # 1. Compute LaBSE Matrix
        e_emb = labse.encode(src.split(), convert_to_tensor=True).cpu().numpy()
        t_emb = labse.encode(tgt.split(), convert_to_tensor=True).cpu().numpy()
        self.labse_mat = cosine_similarity(e_emb, t_emb)

        # 2. Compute MuRIL Matrix
        src_vecs = get_muril_embeddings(src)
        tgt_vecs = get_optimized_embeddings(tgt) # Oops, use local func
        tgt_vecs = get_muril_embeddings(tgt)
        self.muril_mat = cosine_similarity(src_vecs, tgt_vecs)

        self.rows, self.cols = self.labse_mat.shape

print("⏳ Pre-computing embeddings for all 43 pairs...")
CACHE = []
for row in FINAL_DATASET_REFINED:
    CACHE.append(CachedPair(row['eng'], row['tam'], row['indices']))
print("✅ Pre-computation Done.")

# ==========================================================
# 4. GRID SEARCH ENGINE
# ==========================================================
def evaluate_config(alpha, threshold):
    total_aer = 0

    for pair in CACHE:
        # Combine Matrices: Alpha * LaBSE + (1-Alpha) * MuRIL
        ensemble_mat = (pair.labse_mat * alpha) + (pair.muril_mat * (1.0 - alpha))

        # Alignment Logic (Flexible/Iterative)
        pred_links = set()
        for i in range(pair.rows):
            j = np.argmax(ensemble_mat[i])
            if ensemble_mat[i, j] > threshold: pred_links.add((i, j))
        for j in range(pair.cols):
            i = np.argmax(ensemble_mat[:, j])
            if ensemble_mat[i, j] > threshold: pred_links.add((i, j))

        # Calculate AER
        matches = pair.gold_set.intersection(pred_links)
        denominator = len(pair.gold_set) + len(pred_links)
        aer = 1.0 - (2 * len(matches) / denominator) if denominator > 0 else 0.0
        total_aer += aer

    return total_aer / len(CACHE)

# ==========================================================
# 5. RUN GRID SEARCH
# ==========================================================
results = []
alphas = np.arange(0.0, 1.1, 0.1)      # 0.0 to 1.0
thresholds = np.arange(0.3, 0.85, 0.05) # 0.30 to 0.80

print(f"\n🚀 Starting Grid Search ({len(alphas)*len(thresholds)} combinations)...")

best_score = 1.0
best_cfg = None

for alpha in alphas:
    for thresh in thresholds:
        score = evaluate_config(alpha, thresh)
        results.append({
            "LaBSE_Weight": round(alpha, 1),
            "MuRIL_Weight": round(1.0 - alpha, 1),
            "Threshold": round(thresh, 2),
            "Avg_AER": round(score, 4)
        })

        if score < best_score:
            best_score = score
            best_cfg = (alpha, thresh)

# ==========================================================
# 6. RESULTS
# ==========================================================
df_res = pd.DataFrame(results).sort_values(by="Avg_AER")
print("\n🏆 TOP 10 CONFIGURATIONS:")
print(df_res.head(10).to_markdown(index=False))

print(f"\n✅ BEST CONFIG FOUND:")
print(f"   LaBSE Weight : {best_cfg[0]:.1f}")
print(f"   MuRIL Weight : {1.0 - best_cfg[0]:.1f}")
print(f"   Threshold    : {best_cfg[1]:.2f}")
print(f"   Lowest AER   : {best_score:.4f}")

⏳ Loading Models...
✅ Models Ready.
⏳ Pre-computing embeddings for all 43 pairs...
✅ Pre-computation Done.

🚀 Starting Grid Search (121 combinations)...

🏆 TOP 10 CONFIGURATIONS:
|   LaBSE_Weight |   MuRIL_Weight |   Threshold |   Avg_AER |
|---------------:|---------------:|------------:|----------:|
|            0.2 |            0.8 |        0.4  |    0.1443 |
|            0.2 |            0.8 |        0.35 |    0.1521 |
|            0.3 |            0.7 |        0.4  |    0.1532 |
|            0.2 |            0.8 |        0.3  |    0.1547 |
|            0.1 |            0.9 |        0.35 |    0.1583 |
|            0.3 |            0.7 |        0.35 |    0.1612 |
|            0.3 |            0.7 |        0.45 |    0.1616 |
|            0.3 |            0.7 |        0.3  |    0.163  |
|            0.1 |            0.9 |        0.3  |    0.1634 |
|            0.2 |            0.8 |        0.45 |    0.1637 |

✅ BEST CONFIG FOUND:
   LaBSE Weight : 0.2
   MuRIL Weight : 0.8
   Threshol

In [6]:
# @title 🚀 Giga-Aligner V5: Visuals + Text Alignments
# !pip install transformers sentence-transformers scipy scikit-learn

import torch
import numpy as np
import ast
import pandas as pd
from scipy.optimize import linear_sum_assignment
from transformers import AutoTokenizer, AutoModel
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from IPython.display import display, HTML

# ==========================================================
# 1. CONFIGURATION
# ==========================================================
CONFIG = {
    "LABSE_WEIGHT": 0.2,          # Semantic Weight
    "MURIL_WEIGHT": 0.8,          # Syntactic Weight
    "THRESHOLD": 0.4,             # Confidence Threshold
    "USE_STRICT_HUNGARIAN": False # Set False for Flexible (Many-to-One)
}

# ==========================================================
# 2. FINAL CORRECTED GOLD STANDARD (43 Rows)
# ==========================================================
FINAL_DATASET_REFINED = [
    {"eng": "Now, we will look at the overview of Online Social Media.", "tam": "இப்போது நாம் Online Social Media பற்றிய ஒரு மேலோட்டத்தைப் பார்ப்போம்", "indices": "[(0,0), (1,1), (8,2), (9,3), (10,4), (7,5), (5,6), (6,7), (2,8), (3,8), (4,8)]"},
    {"eng": "Social media is of different types, and different types of content are getting generated in our social media.", "tam": "Social media பலவிதமானது, அதில் பலவிதமான உள்ளடக்கங்கள் உருவாகிக் கொண்டிருக்கின்றன", "indices": "[(0,0), (1,1), (4,2), (5,2), (7,4), (8,4), (10,5), (13,6), (11,7), (12,7)]"},
    {"eng": "There are also many different ways in which social media content is generated.", "tam": "மேலும், social media உள்ளடக்கங்கள் உருவாக பல்வேறு வழிகளும் உள்ளன", "indices": "[(2,0), (8,1), (9,2), (10,3), (12,4), (4,5), (5,6), (1,7), (0,7)]"},
    {"eng": "In this course, we will look at the different types of social media services that exist.", "tam": "இந்த course-இல், தற்போது உள்ள பல்வேறு வகையான social media சேவைகளைப் பற்றிப் பார்ப்போம்", "indices": "[(0,1), (1,0), (2,1), (3,0), (8,4), (9,5), (11,6), (12,7), (13,8), (15,3), (4,10), (5,10), (6,10)]"},
    {"eng": "We will also look at how to collect data from social media.", "tam": "மேலும், social media-விலிருந்து தரவுகளை எவ்வாறு சேகரிப்பது என்பது பற்றியும் பார்ப்போம்", "indices": "[(2,0), (10,1), (11,1), (9,1), (8,3), (5,4), (6,5), (7,5), (0,7), (1,7), (3,8), (4,8)]"},
    {"eng": "Specifically, we will look at APIs that we can use to collect data.", "tam": "குறிப்பாக, தரவுகளைச் சேகரிக்க நாம் பயன்படுத்தக்கூடிய API-களைப் பார்ப்போம்", "indices": "[(0,0), (12,1), (10,2), (11,2), (1,3), (7,3), (9,4), (5,5), (2,6), (3,6), (4,6)]"},
    {"eng": "We will specifically look at the Twitter API and the Facebook API.", "tam": "நாம் குறிப்பாக Twitter API மற்றும் Facebook API ஆகியவற்றைப் பார்ப்போம்", "indices": "[(0,0), (2,1), (6,2), (7,3), (8,4), (10,5), (11,6), (1,8), (3,8), (4,8)]"},
    {"eng": "We will see how to collect data from Twitter and Facebook.", "tam": "Twitter மற்றும் Facebook-லிருந்து தரவுகளை எவ்வாறு சேகரிப்பது என்பதை நாம் பார்ப்போம்", "indices": "[(8,0), (9,1), (10,2), (7,2), (6,3), (3,4), (4,5), (5,5), (0,7), (1,8), (2,8)]"},
    {"eng": "We will also look at how to process the collected data.", "tam": "சேகரிக்கப்பட்ட தரவுகளை எவ்வாறு செயல்முறைப்படுத்துவது என்பது பற்றியும் பார்ப்போம்", "indices": "[(9,0), (10,1), (5,2), (7,3), (2,5), (0,6), (1,6), (3,6), (4,6)]"},
    {"eng": "Since the data is text, we will have to look at text analysis techniques.", "tam": "தரவு text வடிவில் இருப்பதால், நாம் text analysis techniques-ஐப் பார்க்க வேண்டும்", "indices": "[(2,0), (4,1), (0,3), (3,3), (5,4), (11,5), (12,6), (13,7), (9,8), (10,8), (6,9), (7,9), (8,9)]"},
    {"eng": "Since the data is in the form of a graph or a network, we will have to use graph analysis techniques.", "tam": "தரவு ஒரு graph அல்லது network வடிவில் இருப்பதால், நாம் graph analysis techniques-ஐப் பயன்படுத்த வேண்டும்", "indices": "[(2,0), (9,2), (10,3), (12,4), (0,6), (13,7), (18,8), (19,9), (20,10), (17,11), (14,12), (15,12), (16,12)]"},
    {"eng": "So we will look at graph analysis metrics like degree, closeness, and betweenness centrality.", "tam": "எனவே, degree, closeness மற்றும் betweenness centrality போன்ற graph analysis metrics-ஐ நாம் பார்ப்போம்", "indices": "[(0,0), (9,1), (10,2), (11,3), (12,4), (13,5), (8,6), (5,7), (6,8), (7,9), (1,10), (2,11), (3,11), (4,11)]"},
    {"eng": "We will also look at how to analyze and visualize networks using a tool called Gephi.", "tam": "Gephi என்ற கருவியைப் பயன்படுத்தி networks-ஐ எவ்வாறு பகுப்பாய்வு செய்வது மற்றும் காட்சிப்படுத்துவது என்பது பற்றியும் பார்ப்போம்", "indices": "[(15,0), (13,2), (11,3), (10,4), (5,5), (7,6), (6,6), (8,8), (9,9), (2,11), (3,11), (1,12), (3,12)]"},
    {"eng": "In the end, we will look at two specific topics.", "tam": "இறுதியாக, நாம் இரண்டு குறிப்பிட்ட தலைப்புகளைப் பார்ப்போம்", "indices": "[(0,0), (1,0), (2,0), (3,1), (7,2), (8,3), (9,4), (4,5), (5,5), (6,5)]"},
    {"eng": "One is privacy, and the other is fake news.", "tam": "ஒன்று தனியுரிமை (privacy), மற்றொன்று போலிச் செய்திகள் (fake news)", "indices": "[(0,0), (2,1), (2,2), (5,3), (7,4), (8,5), (7,6), (8,7)]"},
    {"eng": "The course has tutorials on Python.", "tam": "இந்த course-இல் Python குறித்த tutorials உள்ளன", "indices": "[(0,1), (1,1), (5,2), (4,3), (3,4), (2,5)]"},
    {"eng": "It is not a prerequisite that you should know Python.", "tam": "உங்களுக்கு Python தெரிந்திருக்க வேண்டும் என்பது கட்டாயமில்லை", "indices": "[(6,0), (9,1), (8,2), (7,3), (2,5), (4,5)]"},
    {"eng": "But if you know Python, it will be really useful.", "tam": "ஆனால் உங்களுக்கு Python தெரிந்தால், அது மிகவும் பயனுள்ளதாக இருக்கும்", "indices": "[(0,0), (2,1), (4,2), (3,3), (1,3), (5,4), (9,6), (6,7), (7,7)]"},
    {"eng": "Tutorials on setting up Linux and Python are also uploaded.", "tam": "Linux மற்றும் Python-ஐ நிறுவுவதற்கான tutorials-உம் பதிவேற்றப்பட்டுள்ளன", "indices": "[(4,0), (5,1), (6,2), (2,3), (3,3), (1,3), (0,4), (8,4), (9,5)]"},
    {"eng": "These hands-on tutorials will help students write Python code to collect data from social media platforms", "tam": "இந்த hands-on tutorials மாணவர்களுக்கு Python code எழுதவும், social media platforms-இலிருந்து தரவு சேகரிக்கவும் உதவும்", "indices": "[(0,0), (1,1), (2,2), (5,3), (7,4), (8,5), (6,6), (13,7), (14,8), (15,9), (12,9), (11,10), (10,11), (9,11), (4,12)]"},
    {"eng": "Next week, we will briefly discuss one topic introduced today and provide more hands-on tutorials.", "tam": "அடுத்த வாரம், இன்று அறிமுகப்படுத்தப்பட்ட ஒரு topic-ஐ சிறிது விரிவாகப் பேசுவோம் மற்றும் கூடுதல் hands-on tutorials வழங்கப்படும்", "indices": "[(0,0), (1,1), (8,2), (7,3), (6,4), (5,5), (4,6), (2,7), (3,7), (9,8), (12,9), (13,10), (14,11), (10,12)]"},
    {"eng": "This course is on Online Social Media.", "tam": "இந்த course Online Social Media பற்றியது", "indices": "[(0,0), (1,1), (4,2), (5,3), (6,4), (2,5), (3,5)]"},
    {"eng": "I am Ponnurangam Kumaraguru.", "tam": "நான் பொன்னுரங்கம் குமரகுரு", "indices": "[(0,0), (2,1), (3,2)]"},
    {"eng": "I am a professor at IIIT Hyderabad.", "tam": "நான் IIIT Hyderabad-இல் பேராசிரியராக இருக்கிறேன்", "indices": "[(0,0), (5,1), (6,1), (4,1), (3,2), (1,3)]"},
    {"eng": "We will have weekly sessions where we will look at different topics.", "tam": "நாம் வாராந்திர அமர்வுகளைக் கொண்டிருப்போம், அதில் பல்வேறு தலைப்புகளைப் பார்ப்போம்", "indices": "[(0,0), (3,1), (4,2), (1,3), (2,3), (6,4), (10,5), (11,6), (8,7), (9,7)]"},
    {"eng": "I prepared these slides for a specific purpose.", "tam": "நான் ஒரு குறிப்பிட்ட நோக்கத்திற்காக இந்த slides-ஐ தயார் செய்தேன்", "indices": "[(0,0), (5,1), (6,2), (7,3), (4,3), (2,4), (3,5), (1,6), (1,7)]"},
    {"eng": "This book is very interesting.", "tam": "இந்த புத்தகம் மிகவும் சுவாரஸ்யமாக இருக்கிறது", "indices": "[(0,0), (1,1), (3,2), (4,3), (2,4)]"},
    {"eng": "We went to the market yesterday.", "tam": "நாங்கள் நேற்று சந்தைக்குச் சென்றோம்", "indices": "[(0,0), (5,1), (2,2), (3,2), (4,2), (1,3)]"},
    {"eng": "The cat is sleeping on the mat.", "tam": "பூனை பாயில் தூங்கிக்கொண்டிருக்கிறது", "indices": "[(1,0), (4,1), (5,1), (6,1), (2,2), (3,2)]"},
    {"eng": "He is playing cricket.", "tam": "அவன் கிரிக்கெட் விளையாடிக்கொண்டிருக்கிறான்", "indices": "[(0,0), (3,1), (1,2), (2,2)]"},
    {"eng": "She sings beautifully.", "tam": "அவள் அழகாகப் பாடுகிறாள்", "indices": "[(0,0), (2,1), (1,2)]"},
    {"eng": "They are learning Tamil.", "tam": "அவர்கள் தமிழ் கற்றுக்கொண்டிருக்கிறார்கள்", "indices": "[(0,0), (3,1), (1,2), (2,2)]"},
    {"eng": "The sun rises in the east.", "tam": "சூரியன் கிழக்கில் உதிக்கிறது", "indices": "[(1,0), (3,1), (4,1), (5,1), (2,2)]"},
    {"eng": "I like to drink coffee.", "tam": "எனக்கு காபி குடிக்க பிடிக்கும்", "indices": "[(0,0), (4,1), (2,2), (3,2), (1,3)]"},
    {"eng": "My friend is coming today.", "tam": "என் நண்பன் இன்று வருகிறான்", "indices": "[(0,0), (1,1), (4,2), (2,3), (3,3)]"},
    {"eng": "Please give me some water.", "tam": "தயவுசெய்து எனக்கு கொஞ்சம் தண்ணீர் கொடுங்கள்", "indices": "[(0,0), (2,1), (3,2), (4,3), (1,4)]"},
    {"eng": "We should respect our elders.", "tam": "நாம் பெரியவர்களை மதிக்க வேண்டும்", "indices": "[(0,0), (4,1), (3,1), (2,2), (1,3)]"},
    {"eng": "Education is important for everyone.", "tam": "கல்வி அனைவருக்கும் முக்கியமானது", "indices": "[(0,0), (3,1), (4,1), (2,2), (1,2)]"},
    {"eng": "He runs very fast.", "tam": "அவன் மிகவும் வேகமாக ஓடுகிறான்", "indices": "[(0,0), (2,1), (3,2), (1,3)]"},
    {"eng": "The dog is barking.", "tam": "நாய் குரைத்துக்கொண்டிருக்கிறது", "indices": "[(1,0), (2,1), (3,1)]"},
    {"eng": "I saw a movie yesterday.", "tam": "நான் நேற்று ஒரு படம் பார்த்தேன்", "indices": "[(0,0), (4,1), (2,2), (3,3), (1,4)]"},
    {"eng": "She is reading a book.", "tam": "அவள் ஒரு புத்தகம் படித்துக்கொண்டிருக்கிறாள்", "indices": "[(0,0), (3,1), (4,2), (1,3), (2,3)]"},
    {"eng": "They went to the beach.", "tam": "அவர்கள் கடற்கரைக்குச் சென்றார்கள்", "indices": "[(0,0), (2,1), (3,1), (4,1), (1,2)]"}
]

# ==========================================================
# 3. SETUP MODELS (LaBSE + MuRIL)
# ==========================================================
print("⏳ Loading MuRIL & LaBSE...")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# LaBSE
labse = SentenceTransformer('sentence-transformers/LaBSE').to(device)

# MuRIL
muril_id = "google/muril-base-cased"
tokenizer = AutoTokenizer.from_pretrained(muril_id)
muril = AutoModel.from_pretrained(muril_id, output_hidden_states=True).to(device)
muril.eval()
print("✅ Models Ready.")

# ==========================================================
# 4. ALIGNMENT ENGINE (CombAlign Logic)
# ==========================================================
def get_optimized_embeddings(text):
    words = text.split()
    inputs = tokenizer(words, is_split_into_words=True, return_tensors="pt", padding=True).to(device)
    with torch.no_grad(): out = muril(**inputs)

    # Layer Summation
    hidden_states = torch.stack(out.hidden_states)
    word_embeddings = torch.sum(hidden_states[-4:], dim=0)

    # Subword Pooling
    word_ids = inputs.word_ids()
    final_vectors = []
    current_vectors = []
    current_word_idx = None

    for i, word_idx in enumerate(word_ids):
        if word_idx is None: continue
        if current_word_idx is None:
            current_word_idx = word_idx
            current_vectors.append(word_embeddings[0, i])
        elif word_idx == current_word_idx:
            current_vectors.append(word_embeddings[0, i])
        else:
            final_vectors.append(torch.stack(current_vectors).mean(dim=0))
            current_word_idx = word_idx
            current_vectors = [word_embeddings[0, i]]
    if current_vectors: final_vectors.append(torch.stack(current_vectors).mean(dim=0))
    return torch.stack(final_vectors).cpu().numpy()

def predict_alignment(src, tgt):
    # LaBSE Matrix
    e_emb = labse.encode(src.split(), convert_to_tensor=True).cpu().numpy()
    t_emb = labse.encode(tgt.split(), convert_to_tensor=True).cpu().numpy()
    sim_mat = cosine_similarity(e_emb, t_emb)

    # MuRIL Matrix
    src_vecs = get_optimized_embeddings(src)
    tgt_vecs = get_optimized_embeddings(tgt)
    muril_mat = cosine_similarity(src_vecs, tgt_vecs)

    # Ensemble
    ensemble_mat = (sim_mat * CONFIG["LABSE_WEIGHT"]) + (muril_mat * CONFIG["MURIL_WEIGHT"])
    rows, cols = ensemble_mat.shape
    final_links = set()

    # Iterative Max (Flexible)
    if not CONFIG["USE_STRICT_HUNGARIAN"]:
        for i in range(rows):
            j = np.argmax(ensemble_mat[i])
            if ensemble_mat[i, j] > CONFIG["THRESHOLD"]: final_links.add((i, j))
        for j in range(cols):
            i = np.argmax(ensemble_mat[:, j])
            if ensemble_mat[i, j] > CONFIG["THRESHOLD"]: final_links.add((i, j))
    else:
        # Hungarian (Strict)
        if rows == cols:
            row_ind, col_ind = linear_sum_assignment(1.0 - ensemble_mat)
            for r, c in zip(row_ind, col_ind): final_links.add((r, c))

    return sorted(list(final_links))

# ==========================================================
# 5. TEXT & VISUALIZATION HELPERS
# ==========================================================
def calculate_aer(gold_indices_str, pred_links):
    gold_set = set(ast.literal_eval(gold_indices_str))
    pred_set = set(pred_links)
    matches = gold_set.intersection(pred_set)
    numerator = 2 * len(matches)
    denominator = len(gold_set) + len(pred_set)
    if denominator == 0: return 0.0
    return 1.0 - (numerator / denominator)

def get_text_alignment(src_text, tgt_text, pred_links):
    src_words = src_text.split()
    tgt_words = tgt_text.split()
    pairs = []

    for i, j in pred_links:
        if i < len(src_words) and j < len(tgt_words):
            pairs.append(f"{src_words[i]} ↔ {tgt_words[j]}")

    return " , ".join(pairs)

def render_prediction(src_text, tgt_text, gold_indices_str, pred_links, aer, idx):
    src_words = src_text.split()
    tgt_words = tgt_text.split()
    text_alignments = get_text_alignment(src_text, tgt_text, pred_links)

    width = 800
    height = 180
    src_gap = (width - 100) / max(1, len(src_words) - 1)
    tgt_gap = (width - 100) / max(1, len(tgt_words) - 1)

    color = "green" if aer < 0.3 else ("orange" if aer < 0.5 else "red")

    html = f"""
    <div style="font-family: Arial; border: 1px solid #ddd; padding: 15px; margin-bottom: 20px; background: white; border-radius: 8px;">
        <div style="font-weight: bold; color: #333; margin-bottom: 10px; border-bottom: 1px solid #eee; padding-bottom: 5px;">
            Row #{idx} | AER: <span style="color:{color}">{aer:.2f}</span>
        </div>
        <div style="font-size: 13px; color: #555; margin-bottom: 10px; background: #f9f9f9; padding: 8px; border-radius: 4px;">
            {text_alignments}
        </div>
        <svg width="{width}" height="{height}">
    """

    src_x = [50 + i * src_gap for i in range(len(src_words))]
    tgt_x = [50 + i * tgt_gap for i in range(len(tgt_words))]

    # Draw Links
    for i, j in pred_links:
        if i < len(src_x) and j < len(tgt_x):
            html += f'<path d="M{src_x[i]},50 C{src_x[i]},120 {tgt_x[j]},110 {tgt_x[j]},140" stroke="#007bff" stroke-width="2" fill="none" opacity="0.6"/>'

    # Draw Words
    for i, w in enumerate(src_words):
        html += f'<text x="{src_x[i]}" y="40" text-anchor="middle" font-size="13" font-family="Arial" fill="#2c3e50">{w}</text>'
        html += f'<circle cx="{src_x[i]}" cy="50" r="3" fill="#2c3e50"/>'

    for i, w in enumerate(tgt_words):
        html += f'<text x="{tgt_x[i]}" y="155" text-anchor="middle" font-size="13" font-family="Arial" fill="#e91e63">{w}</text>'
        html += f'<circle cx="{tgt_x[i]}" cy="140" r="3" fill="#e91e63"/>'

    html += "</svg></div>"
    return HTML(html)

# ==========================================================
# 6. RUN AND DISPLAY ALL
# ==========================================================
print("\n📊 VISUALIZING MODEL PREDICTIONS vs GOLD STANDARD...")
print(f"Algorithm: CombAlign (MuRIL + LaBSE) | Weights: {CONFIG['LABSE_WEIGHT']}/{CONFIG['MURIL_WEIGHT']}")
print("-" * 50)

total_aer = 0
for i, row in enumerate(FINAL_DATASET_REFINED):
    pred_links = predict_alignment(row['eng'], row['tam'])
    aer = calculate_aer(row['indices'], pred_links)
    total_aer += aer

    display(render_prediction(
        row['eng'],
        row['tam'],
        row['indices'],
        pred_links,
        aer,
        i + 1
    ))

print(f"\n🔴 OVERALL AVERAGE AER: {total_aer / len(FINAL_DATASET_REFINED):.4f}")

⏳ Loading MuRIL & LaBSE...
✅ Models Ready.

📊 VISUALIZING MODEL PREDICTIONS vs GOLD STANDARD...
Algorithm: CombAlign (MuRIL + LaBSE) | Weights: 0.2/0.8
--------------------------------------------------



🔴 OVERALL AVERAGE AER: 0.1576
